In [1]:
from datetime import datetime
import hashlib
from pathlib import Path

def base_metadata(file_path: str, source_type: str):
    return {
        "source_type": source_type,
        "file_name": Path(file_path).name,
        "file_path": str(file_path),
        "ingested_at": datetime.utcnow().isoformat()
    }

def hash_text(text: str):
    return hashlib.md5(text.encode("utf-8")).hexdigest()


In [2]:
from llama_index.readers.file import PDFReader
from llama_index.core import SimpleDirectoryReader

pdf_reader = PDFReader()

pdf_docs = SimpleDirectoryReader(
    input_dir="../data/raw",
    file_extractor={".pdf": pdf_reader},
    recursive=True
).load_data()

clean_pdf_docs = []

for doc in pdf_docs:
    if len(doc.text.strip()) < 100:
        continue  # drop junk pages

    meta = base_metadata(doc.metadata.get("file_path", "unknown"), "pdf")
    meta["content_hash"] = hash_text(doc.text)

    doc.metadata = meta
    clean_pdf_docs.append(doc)

print(f"Clean PDF docs: {len(clean_pdf_docs)}")


Clean PDF docs: 15


In [ ]:
# from llama_index.core import Document
# import pandas as pd

# df = pd.read_csv("../data/raw/sample.csv")

# csv_docs = []

# for idx, row in df.iterrows():
#     text = f"""
#     Record from sample.csv

#     {row.to_json()}
#     """.strip()

#     meta = base_metadata("../data/raw/sample.csv", "csv")
#     meta.update({
#         "row_index": idx,
#         "content_hash": hash_text(text)
#     })

#     csv_docs.append(Document(text=text, metadata=meta))

# print(f"CSV docs: {len(csv_docs)}")


In [3]:
all_documents = clean_pdf_docs  #+ csv_docs
print("Total documents:", len(all_documents))


Total documents: 15


In [4]:
import pickle
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

with open("../data/processed/all_documents.pkl", "wb") as f:
    pickle.dump(all_documents, f)

print("Saved all_documents.pkl successfully")


Saved all_documents.pkl successfully
